In [3]:
from sympy import *

def prolongation(vf, variables, order):
    """
    Compute the prolongation of a vector field.
    
    :param vf: vector field as a list of sympy expressions
    :param variables: list of sympy symbols for the independent variables
    :param order: order of prolongation
    :return: prolonged vector field as a list of sympy expressions
    """
    # Create the dependent variable and its derivatives
    from sympy.utilities.iterables import multiset_partitions
    dep_var = Function('u')(*variables)
    #derivatives = [dep_var] + [Derivative(dep_var, *multi_index) 
    #                           for i in range(1, order + 1)
    #                           for multi_index in multiset_partitions(variables, i)]

    derivatives = [dep_var]
    from more_itertools import flatten
    for i in range(1, order + 1):
        for multi_index in multiset_partitions(variables, i):
            print(f"{locals()=}")
            derivatives.append(Derivative(dep_var, *flatten(multi_index)))
    
    # Compute the prolongation
    prolonged_vf = []
    for component in vf:
        prolonged_component = component
        for deriv in derivatives:
            prolonged_component += sum(diff(component, var) * deriv.diff(var) 
                                       for var in variables)
        prolonged_vf.append(prolonged_component)
    
    return prolonged_vf

# Example usage
x, y = symbols('x y')
vf = [x * Function('u')(x, y), y * Function('u')(x, y)]
variables = [x, y]
order = 2

prolonged_vf = prolongation(vf, variables, order)
for component in prolonged_vf:
    pprint(component)


locals()={'vf': [x*u(x, y), y*u(x, y)], 'variables': [x, y], 'order': 2, 'multiset_partitions': <function multiset_partitions at 0x7ac992b5a0c0>, 'dep_var': u(x, y), 'derivatives': [u(x, y)], 'flatten': <function flatten at 0x7ac991b7e840>, 'i': 1, 'multi_index': [[x, y]]}
locals()={'vf': [x*u(x, y), y*u(x, y)], 'variables': [x, y], 'order': 2, 'multiset_partitions': <function multiset_partitions at 0x7ac992b5a0c0>, 'dep_var': u(x, y), 'derivatives': [u(x, y), Derivative(u(x, y), x, y)], 'flatten': <function flatten at 0x7ac991b7e840>, 'i': 2, 'multi_index': [[x], [y]]}
                           2                      3                            ↪
              ⎛∂          ⎞        ∂             ∂               ⎛  ∂          ↪
x⋅u(x, y) + x⋅⎜──(u(x, y))⎟  + 2⋅x⋅──(u(x, y))⋅──────(u(x, y)) + ⎜x⋅──(u(x, y) ↪
              ⎝∂y         ⎠        ∂y            2               ⎝  ∂x         ↪
                                               ∂y  ∂x                          ↪

↪                

In [2]:
from sympy import symbols, Poly
from sympy.polys.monomialtools import monomial_div

def leading_term(poly, order):
    terms = list(poly.terms(order=order))
    return terms[0]

def monomial_order(monomial, order):
    return tuple(monomial[var] for var in order)

def janet_basis(polys, order):
    basis = []
    for poly in polys:
        lt = leading_term(poly, order)
        basis.append(poly / lt[0])
    return basis

def example():
    x, y, z = symbols('x y z')
    polys = [Poly(x**2 + y**2 - 1, x, y, z), Poly(x**2 + z**2 - 1, x, y, z)]
    order = [x, y, z]
    basis = janet_basis(polys, order)
    for b in basis:
        print(b)

if __name__ == "__main__":
    example()

ModuleNotFoundError: No module named 'sympy.polys.monomialtools'

In [5]:
from sympy import symbols, expand, poly
#from sympy.core.compatibility import reduce
from sympy.polys.orderings import monomial_key
from sympy.polys.monomials import itermonomials

def janet_basis(groebner_basis, order):
    """
    Compute the Janet basis of a Groebner basis using the specified monomial order.

    Parameters:
    - groebner_basis: List of polynomials forming a Groebner basis.
    - order: Monomial order to be used.

    Returns:
    - List of polynomials forming the Janet basis.
    """
    def janet_divisors(monomial, variables, order):
        divisors = []
        for var in variables:
            if monomial.diff(var) != 0:
                divisors.append(monomial_key(order, [var])[0])
        return divisors

    def reduce_modulo(f, G):
        for g in G:
            while f != 0 and g != 0 and f.lm % g.lm == 0:
                f -= (f.lc / g.lc) * g
        return f

    def update_basis(F, G):
        for f in F:
            h = reduce_modulo(f, G)
            if h != 0:
                G.append(h)
        return G

    variables = groebner_basis[0].gens
    G = [poly(g, *variables, order=order) for g in groebner_basis]
    monomials = list(itermonomials(variables, order))

    for m in monomials:
        divisors = janet_divisors(m, variables, order)
        F = [m * g for g in G if any(m.diff(d) != 0 for d in divisors)]
        G = update_basis(F, G)

    return G

def example_code():
    x, y, z = symbols('x y z')
    polynomials = [x**2 + y**2 - 1, x**2 - z]
    orders = ['lex', 'grlex', 'grevlex']

    for order in orders:
        basis = janet_basis(polynomials, order)
        print(f'Janet basis for {order} order:')
        for poly in basis:
            print(poly)

if __name__ == "__main__":
    example_code()

AttributeError: 'Add' object has no attribute 'gens'